<a href="https://colab.research.google.com/github/Ravi-ranjan1801/CUDA-Lab/blob/main/cuda_lab_6_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile word_count.cu
// word_count.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

#define MAX_WORDS 1000
#define MAX_LEN 20

__global__ void countWords(char words[][MAX_LEN], int *counts, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if(i < n)
    {
        int cnt = 0;

        for(int j=0;j<n;j++)
        {
            int same = 1;

            for(int k=0;k<MAX_LEN;k++)
            {
                if(words[i][k] != words[j][k])
                {
                    same = 0;
                    break;
                }
            }

            if(same) cnt++;
        }

        counts[i] = cnt;
    }
}

int main()
{
    FILE *fp = fopen("input.txt","r");

    if(fp == NULL)
    {
        printf("File not found\n");
        return 0;
    }

    char words[MAX_WORDS][MAX_LEN];
    int n = 0;

    // Read words
    while(fscanf(fp,"%s",words[n]) != EOF)
    {
        // convert to lowercase
        for(int i=0;words[n][i];i++)
            words[n][i] = tolower(words[n][i]);

        n++;
    }

    fclose(fp);

    // Device memory
    char (*d_words)[MAX_LEN];
    int *d_counts;

    cudaMalloc(&d_words, sizeof(words));
    cudaMalloc(&d_counts, n*sizeof(int));

    cudaMemcpy(d_words, words, sizeof(words), cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks = (n + threads -1)/threads;

    countWords<<<blocks,threads>>>(d_words, d_counts, n);

    int counts[MAX_WORDS];
    cudaMemcpy(counts, d_counts, n*sizeof(int), cudaMemcpyDeviceToHost);

    // Print distinct words
    printf("\nWord Count:\n");

    for(int i=0;i<n;i++)
    {
        int printed = 0;

        for(int j=0;j<i;j++)
        {
            if(strcmp(words[i],words[j])==0)
            {
                printed = 1;
                break;
            }
        }

        if(!printed)
            printf("%s : %d\n", words[i], counts[i]);
    }

    cudaFree(d_words);
    cudaFree(d_counts);

    return 0;
}

Writing word_count.cu


In [2]:
%%writefile input.txt
hello world hello cuda gpu world cuda cuda

Writing input.txt


In [3]:
!nvcc word_count.cu -o wc
!./wc

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).

Word Count:
hello : 2
world : 2
cuda : 3
gpu : 1
